<a href="https://colab.research.google.com/github/brianellis1997/Sarcasm_Detection/blob/main/DS_440_Traditional_Models_Logit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sarcasm Detection using NLP Techniques
### Team 2: Brian Ellis, Lindsey Rich, Elena Kern

## Load Libraries and Dataset

In [ ]:
import pandas as pd
import torch
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#test comment

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

In [ ]:
!git clone https://github.com/brianellis1997/Sarcasm_Detection.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

In [ ]:
test = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')

In [ ]:
train_bal.info()

In [ ]:
train_bal['label'].value_counts()

We can see our dataset is balanced.

In [ ]:
train_bal['label'].value_counts(normalize=True)

## EDA (WordCloud)

In [ ]:
# # Separating sarcastic vs non sarcastic comments
# sarcastic_com = train_bal[train_bal['label'] == 1]
# non_sarcastic_com = train_bal[train_bal['label'] == 0]

In [ ]:
# import nltk
# from nltk import word_tokenize, sent_tokenize
# from nltk.corpus import stopwords
# from nltk.stem.porter import *
# nltk.download('punkt')
# nltk.download('stopwords')

# import string

# import gensim
# from gensim.models.phrases import Phraser, Phrases
# from gensim.models.word2vec import Word2Vec

# import spacy # for a lemmatization example

# from sklearn.manifold import TSNE

# import pandas as pd
# from bokeh.io import output_notebook, output_file
# from bokeh.plotting import show, figure

In [ ]:
# # Remove stopwords and stem to group together similar words
# stpwrds = stopwords.words('english') + list(string.punctuation)
# stpwrds

In [ ]:
# for s in comments[:10]:
#   print(s)

In [ ]:
# import nltk
# from nltk.corpus import stopwords

# # Download stopwords if not already downloaded
# nltk.download('stopwords')

# # Get the English stopwords
# stpwrds = set(stopwords.words('english'))

# # Assuming 'comment' is the column containing the sentences in your DataFrame
# comments = train_bal['comment']

# # Function to tokenize, lowercase, and remove stopwords from each comment
# def clean_comments(comments):
#     clean_comments_list = []
#     for comment in comments:
#         # Tokenize and lowercase
#         tokens = comment.lower().split()

#         # Remove stopwords
#         cleaned_tokens = [token for token in tokens if token not in stpwrds]

#         clean_comments_list.append(cleaned_tokens)

#     return clean_comments_list

In [ ]:
# sarcastic_comments = sarcastic_com['comment']
# non_sarcastic_comments = non_sarcastic_com['comment']

# # Apply the function to sarcastic and non-sarcastic comments
# sarcastic_clean = clean_comments(sarcastic_comments)
# non_sarcastic_clean = clean_comments(non_sarcastic_comments)

# # Example output for the first 10 sarcastic comments
# print(sarcastic_clean[:10])

In [ ]:
# from gensim.models.phrases import Phraser, Phrases

# # Add common two-word phrases aka Bi-grams
# sarc_bigrams = Phraser(Phrases(sarcastic_clean, min_count=32, threshold=64))
# non_sarc_bigrams = Phraser(Phrases(non_sarcastic_clean, min_count=32, threshold=64))

# # Apply the bigram models to the tokenized lists
# clean_sarc = [sarc_bigrams[sentence] for sentence in sarcastic_clean]
# clean_non_sarc = [non_sarc_bigrams[sentence] for sentence in non_sarcastic_clean]

# # Example output for the first 10 sarcastic comments after adding bigrams
# print(clean_sarc[:10])

In [ ]:
# # WordCloud for sarcastic comments
# wordcloud_sarc = WordCloud(width=800, height=400, background_color='white').generate(' '.join([' '.join(sentence) for sentence in clean_sarc]))

# # Display the generated word cloud using matplotlib
# plt.figure(figsize=(10, 5))
# plt.imshow(wordcloud_sarc, interpolation='bilinear')
# plt.axis('off')
# plt.title('Word Cloud for Sarcastic Comments')
# plt.show()


In [ ]:
# # WordCloud for sarcastic comments
# wordcloud_sarc = WordCloud(width=800, height=400, background_color='white').generate(' '.join([' '.join(sentence) for sentence in clean_non_sarc]))

# # Display the generated word cloud using matplotlib
# plt.figure(figsize=(10, 5))
# plt.imshow(wordcloud_sarc, interpolation='bilinear')
# plt.axis('off')
# plt.title('Word Cloud for Non-Sarcastic Comments')
# plt.show()

# Visualizing Sarcastic Comments by Subreddit

In [ ]:
# subreddit_count = sarcastic_com['subreddit'].value_counts()

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# top = subreddit_count.head(20)
# plt.figure(figsize=(10, 6))
# sns.barplot(x=top.values, y=top.index)
# plt.xlabel('Number of Sarcastic Comments')
# plt.ylabel('Subreddit')
# plt.title('Frequency of Sarcastic Comments by Subreddit')
# plt.show()


## Train Logistic Regression model (baseline)

In [ ]:
from sklearn.model_selection import train_test_split
comments = train_bal['comment']
# Create a new column 'merged_comment' by concatenating 'comment' and 'parent_comment'
train_bal['merged_comment'] = train_bal['comment'] + ' ' + train_bal['parent_comment']

labels = train_bal['label']

X_train, X_test, y_train, y_test = train_test_split(train_bal['merged_comment'], labels, test_size=0.3, random_state = 42)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# build bigrams
# and minimal word frequency
tf_idf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)
# multinomial logistic regression a.k.a softmax classifier
logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=42, verbose=1)
# sklearn's pipeline
tfidf_logit_pipeline = Pipeline([('tf_idf', tf_idf),
                                 ('logit', logit)])

In [ ]:
%%time
tfidf_logit_pipeline.fit(X_train, y_train)

In [ ]:
%%time
valid_pred = tfidf_logit_pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, valid_pred)

## Stacking with parent comment

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)

comment_vectors = tf_idf_comment.fit_transform(train_bal['comment'])
parent_comment_vectors = tf_idf_parent_comment.fit_transform(train_bal['parent_comment'])

# Concatenate the TF-IDF vectors horizontally
combined_vectors = hstack([comment_vectors, parent_comment_vectors])

In [ ]:
# Inspect TF-IDF vectors and their size
print("TF-IDF vectors for comments:")
print(comment_vectors)
print("Size of comment_vectors:", comment_vectors.shape)

print("\nTF-IDF vectors for parent comments:")
print(parent_comment_vectors)
print("Size of parent_comment_vectors:", parent_comment_vectors.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

labels = train_bal['label']
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(combined_vectors, labels, test_size=0.2, random_state=22)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
valid_pred = logit.predict(X_test)

from sklearn.metrics import accuracy_score
accuracy_score(y_test, valid_pred)

## Using non-text covariates

In [ ]:
train_bal.info()

### Author

In [ ]:
train_bal['author'].nunique()

In [ ]:
train_bal['subreddit'].nunique()

In [ ]:
train_bal.head()

In [ ]:
# Sarcastic authors
train_bal[train_bal['label'] == 1].groupby('author').size().sort_values(ascending=False).head(5)

In [ ]:
# Least sarcastic authors
train_bal[train_bal['label'] == 0].groupby('author').size().sort_values(ascending=False).head(5)

It seems that the authors with the most sarcastic comments are also the ones with the most non-sarcastic comments. Maybe author isn't a valuable predictor for sarcasm. We can test this by performing a chi-squared test.

In [ ]:
from scipy.stats import chi2_contingency

# Create a contingency table
contingency_table = pd.crosstab(train_bal['author'], train_bal['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

With our statistical test, we see that there is very little correlation between these variables, this means that the author variable might add unwanted noise to our model and might not be worth including.

### Subreddit

In [ ]:
# Sarcastic subreddits
train_bal[train_bal['label'] == 1].groupby('subreddit').size().sort_values(ascending=False).head(5)

In [ ]:
# Non-Sarcastic subreddits
train_bal[train_bal['label'] == 0].groupby('subreddit').size().sort_values(ascending=False).head(5)

In [ ]:
train_bal['subreddit'].nunique()

In [ ]:
top_subreddits = train_bal['subreddit'].value_counts(ascending=False).head(10).index
subreddit_filter = train_bal[train_bal['subreddit'].isin(top_subreddits)]

subreddit_filter.info()

In [ ]:
sns.countplot(x='subreddit', hue='label', data=subreddit_filter)
plt.title('Sarcasm by Subreddit')
plt.xlabel('Subreddits')
plt.ylabel('Count')
plt.legend(labels=['No Sarcasm', 'Sarcasm'])
plt.xticks(rotation=90)
plt.show()

We get a promising plot that our covariate `subreddit` may have some influence on our sarcasm label. Let's test this with a chi-squared test.

In [ ]:
# Create a contingency table
contingency_table = pd.crosstab(train_bal['subreddit'], train_bal['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

We get a statistically significant result with a p-value of 0. Let's test this with our reduced subreddit dataset.

In [ ]:
# Create a contingency table
contingency_table = pd.crosstab(subreddit_filter['subreddit'], subreddit_filter['label'])

# Perform chi-square test
chi2, p, _, _ = chi2_contingency(contingency_table)
print(f"Chi-square value: {chi2}\nP-value: {p}")

Still a significant value, this means `subreddit` is a valuable feature for predicting sarcasm.

### Up and Down Votes

In [ ]:
# Filter dataset
Votes = train_bal[['score', 'ups', 'downs', 'label']]
Votes.info()

In [ ]:
from scipy.stats import pearsonr

voting_corr = Votes.corr()

# Create an empty DataFrame to store p-values
p_values = pd.DataFrame(index=voting_corr.index, columns=voting_corr.columns)

# Loop through pairs of variables and calculate correlation and p-value
for col1 in voting_corr.columns:
    for col2 in voting_corr.index:
        corr, p_value = pearsonr(Votes[col1], Votes[col2])
        p_values.loc[col1, col2] = p_value

In [ ]:
train_bal.describe()

In [ ]:
train_bal['ups'].median()

In [ ]:
voting_corr

In [ ]:
p_values

In [ ]:
plt.hist(Votes['downs'])
plt.title('Down Vote Distribution')
plt.show()

In [ ]:
Votes['ups'].nunique()

In [ ]:
Votes['ups'].max()

In [ ]:
import numpy as np

plt.hist(Votes['ups'])
plt.title('Up Vote Distribution')
plt.show()

In [ ]:
plt.hist(Votes['score'])
plt.title('Distribution of Score')
plt.show()

In [ ]:
train_bal.info()

Maybe train a model only with voting variables to see influence?

### Date

In [ ]:
train_bal['date'].head()

In [ ]:
train_bal['date'] = pd.to_datetime(train_bal['date']) # Run this to convert date to date_time
train_bal['date'].head()

In [ ]:
train_bal['created_utc'].head()

`date` and `created_utc` may hold significance in our sarcastic prediction task. Maybe sarcastic comments are more prevelant during certain times of the year and maybe more prevelant during times of the day as well.

#### Month

In [ ]:
monthly_counts = train_bal.groupby(['label', train_bal['date'].dt.month])['date'].count().unstack()
monthly_counts

In [ ]:
monthly_counts.columns

In [ ]:
# Melt the DataFrame to have a 'month' column
monthly_counts_melted = monthly_counts.reset_index().melt(id_vars=['label'], var_name='month', value_name='count')

# Plot the counts using a bar plot
sns.barplot(x='month', y='count', hue='label', data=monthly_counts_melted)
plt.title('Counts of Each Label for Each Month')
plt.xlabel('Month')
plt.ylabel('Count')
plt.show()

It seems that there are more sarcastic comments at the beginning of the year and more non-sarcastic comments towards the end of the year.<br>We can perform a statistical test to see if there's a relationship between month and sarcasm label.

In [ ]:
from scipy.stats import chi2_contingency
# Create a contingency table
contingency_table = monthly_counts_melted.pivot_table(values='count', index='month', columns='label', fill_value=0)

contingency_table

In [ ]:
# Perform chi-square test
chi2, p_value, _, _ = chi2_contingency(contingency_table)

# Print the result
print(f"Chi-square value: {chi2}")
print(f"P-value: {p_value}")

There is a statistically significant relationship between months and sarcasm counts. <br>This tells us that including the `month` variable into our dataframe might be good for prediction.

#### Year

In [ ]:
yearly_counts = train_bal.groupby(['label', train_bal['date'].dt.year])['date'].count().unstack()
yearly_counts

In [ ]:
# Melt the DataFrame to have a 'month' column
yearly_counts_melted = yearly_counts.reset_index().melt(id_vars=['label'], var_name='year', value_name='count')

# Plot the counts using a bar plot
sns.barplot(x='year', y='count', hue='label', data=yearly_counts_melted)
plt.title('Counts of Each Label for Each Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
# Create a contingency table
contingency_table = yearly_counts_melted.pivot_table(values='count', index='year', columns='label', fill_value=0)

contingency_table

In [ ]:
# Perform chi-square test
chi2, p_value, _, _ = chi2_contingency(contingency_table)

# Print the result
print(f"Chi-square value: {chi2}")
print(f"P-value: {p_value}")

I don't know if `year` would have the same predictive power as `month` because it is not cyclic and has much less data.

### UTC

In [ ]:
train_bal['created_utc'] = pd.to_datetime(train_bal['created_utc']) # Run this to convert created_utc to date_time
train_bal.info()

In [ ]:
train_bal['hour'] = train_bal['created_utc'].dt.hour # Run this to feature engineer hour variable

In [ ]:
sns.countplot(x='hour', hue='label', data=train_bal)
plt.title('Distribution of Sarcastic Comments by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Count')
plt.legend(title='Label', labels=['Non-Sarcastic', 'Sarcastic'])
plt.show()

Interestingly enough, it seems like there are more sarcastic comments in the evening, around the time people come back from work/school. There are less sarcastic comments in the morning.

In [ ]:
train_bal['date'].dt.day_of_week.nunique()

In [ ]:
train_bal['month'] = train_bal['date'].dt.month # Run this cell to add month and day of week variables
# train_bal['year'] = train_bal['date'].dt.year
train_bal['dow'] = train_bal['date'].dt.day_of_week
train_bal.info()

In [ ]:
sns.countplot(x='dow', hue='label', data=train_bal)
plt.title('Distribution of Sarcastic Comments by Day of Week')
plt.xlabel('Day of the Week')
plt.ylabel('Count')
plt.legend(title='Label', labels=['Non-Sarcastic', 'Sarcastic'])
plt.show()

In [ ]:
# Group by hour, month_name, and label, and calculate the count
grouped_time = train_bal.groupby(['hour', 'month', 'label']).size().reset_index(name='count')

grouped_time

After doing this analysis, it seems like month and hour would be valuable variables to include in our model because they have variance with the target label, i.e. sarcastic comments vary by time.

## Logistic Regression (non-text covariates)

---
We plan on building an initial model with all variables included to see a a few things: performance, runtime, and feature importance.<br>If the model runs slow, we might consider pruning our dataset, if it performs poorly we might perform feature selection.<br>We have initial theories as to which variables are most important based on their variance with the target label but we want to implement a model with all the data initially.


---



### Data preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
train_bal.info()

In [ ]:
# Drop unneeded columns
train = train_bal.drop(columns=['Unnamed: 0', 'date', 'created_utc', 'author', 'comment', 'parent_comment'])
train.info()

In [ ]:
# # One-hot encode subreddit
# subreddit_dummies = pd.get_dummies(train_bal['subreddit'])

In [ ]:
# # Combine one-hot encoded categorical variables with numeric variables
# train_encoded = pd.concat([train, subreddit_dummies], axis=1)
# train_encoded.info()

The code above crashed our google colab because of lack of RAM. We might need to filter our subreddits.

We have 3 potential options:
<li>Pay for more compute in google colab</li>
<li>Locally merge subreddit dummies into train df and upload to colab</li>
<li>Try computing on ROAR</li>
<li>Saving dummies into file and then concatenate</li>
<li>Batch append variables (for loop)</li>
<li>Maybe group subreddits</li>

In [ ]:
# # Define the batch size
# batch_size = 10  # Adjust this value based on your available memory and desired batch size

# # Calculate the number of batches
# num_batches = len(subreddit_dummies) // batch_size + 1

# # Concatenate in batches
# concatenated_dfs = []
# for i in range(num_batches):
#     start_idx = i * batch_size
#     end_idx = (i + 1) * batch_size
#     if end_idx > len(subreddit_dummies):
#         end_idx = len(subreddit_dummies)
#     subset_dummies = subreddit_dummies.iloc[start_idx:end_idx]
#     concatenated_df = pd.concat([train, subset_dummies], axis=1)
#     concatenated_dfs.append(concatenated_df)

# # Concatenate all batches
# train_concat = pd.concat(concatenated_dfs)

Concatenating in batch sizes also didn't work which leads me to believe it is a problem concatenating the amount of columns. Therefore, I propose we keep the top 50 or so subreddits with the most comments and assign the remaining subreddits to one single group `other`.

In [ ]:
train['subreddit'].nunique() # 13,561 different subreddits

In [ ]:
subreddit_counts = train_bal.groupby('subreddit').size().sort_values(ascending=False)
subreddit_counts.head()

In [ ]:
# Calculate the 5% quantile
quantile_5 = subreddit_counts.quantile(0.95)

# Select subreddits whose counts are greater than or equal to the 5% quantile
top_subreddits = subreddit_counts[subreddit_counts >= quantile_5]
top_subreddits.size # 680 subreddits in the top 5%

In [ ]:
top_subreddits.tail()

In [ ]:
subreddits = top_subreddits.index.tolist()
subreddits[:5]

In [ ]:
def filter_subreddit(subreddit):
    if subreddit in subreddits:
        return subreddit
    else:
        return 'other'

# Apply the filter_subreddit function to the 'subreddit' column
train['subreddit'] = train['subreddit'].apply(filter_subreddit)

train['subreddit'].nunique()

In [ ]:
# Most popular subreddits
train.groupby('subreddit').size().sort_values(ascending=False).head(10)

In [ ]:
top_subreddits = train['subreddit'].value_counts(ascending=False).head(10).index
subreddit_filter = train[train['subreddit'].isin(top_subreddits)]

subreddit_filter.info()

In [ ]:
sns.countplot(x='subreddit', hue='label', data=subreddit_filter)
plt.title('Sarcasm by Subreddit')
plt.xlabel('Subreddits')
plt.ylabel('Count')
plt.legend(labels=['No Sarcasm', 'Sarcasm'])
plt.xticks(rotation=90)
plt.show()

It seems like the lesser-used subreddits contain more non-sarcastic comments.

In [ ]:
# One-hot encode subreddit
subreddit_dummies = pd.get_dummies(train['subreddit'])

In [ ]:
# Combine one-hot encoded categorical variables with numeric variables
train_encoded = pd.concat([train, subreddit_dummies], axis=1)
train_encoded.info()

In [ ]:
train_encoded.drop(columns='subreddit', inplace=True)

In [ ]:
train_encoded.shape

In [ ]:
train_encoded.head()

In [ ]:
X = train_encoded.drop(columns='label')
y = train_encoded['label']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=22)

print(X_train.shape)
print(X_val.shape)

In [ ]:
sc = StandardScaler() # Scale variables

X_train = sc.fit_transform(X_train)
X_val = sc.transform(X_val)

print(X_train.shape)
print(X_val.shape)

In [ ]:
logit = LogisticRegression(random_state=22)

logit.fit(X_train, y_train)

y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

## Baseline performance with Comment only

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_val, y_train, y_val = train_test_split(train_bal['comment'], train_bal['label'], test_size=0.2, random_state=22)
print(X_train.shape)
print(X_val.shape)

In [ ]:
# build bigrams, put a limit on maximal number of features
# and minimal word frequency
tf_idf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2)
# multinomial logistic regression a.k.a softmax classifier
logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=17, verbose=1)
# sklearn's pipeline
tfidf_logit_pipeline = Pipeline([('tf_idf', tf_idf),
                                 ('logit', logit)])

In [ ]:
%%time
tfidf_logit_pipeline.fit(X_train, y_train)

In [ ]:
%%time
y_pred = tfidf_logit_pipeline.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

### Inference

In [ ]:
!pip install eli5

This code will display the weights associated with each feature (word) in our logistic regression model. It will help us understand which words/phrases contribute most to the prediction of sarcasm/non-sarcasm.

In [ ]:
import eli5
eli5.show_weights(estimator=tfidf_logit_pipeline.named_steps['logit'],
                  vec=tfidf_logit_pipeline.named_steps['tf_idf'])

## Train Logit with text and non-text covars

In [ ]:
import pandas as pd

train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

In [ ]:
# train_bal['exclamation_count'] = train_bal['comment'].apply(lambda x: x.count('!'))


# train_bal['questionmark_count'] = train_bal['comment'].apply(lambda x: x.count('?'))


# train_bal['period_count'] = train_bal['comment'].apply(lambda x: x.count('.'))


# train_bal['space_count'] = train_bal['comment'].apply(lambda x: x.count('_'))


# train_bal['slash_count'] = train_bal['comment'].apply(lambda x: x.count('/'))


# train_bal['elips_count'] = train_bal['comment'].apply(lambda x: x.count(':'))


# import pandas as pd

# train_bal['capital_letters_count'] = train_bal['comment'].apply(lambda x: sum(1 for char in x if char.isupper()))



# train_bal['word_count'] = train_bal['comment'].apply(lambda x: len(x.split()))


# import pandas as pd
# import string

# def count_punctuation(comment):
#     return sum(1 for char in comment if char in string.punctuation)


# train_bal['total_punctuation'] = train_bal['comment'].apply(count_punctuation)

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Define keywords related to news or politics
# news_politics_keywords = ['news', 'politics', 'worldnews', 'worldpolitics', 'uspolitics', 'worldnews', 'worldpolitics']

# # Calculate the count of occurrences of each subreddit
# subreddit_counts = train_bal['subreddit'].value_counts()

# # Filter DataFrame for subreddits with more than 100 occurrences and containing keywords
# news_politics_comments = train_bal[train_bal['subreddit'].isin(subreddit_counts[subreddit_counts > 100].index) &
#                                     train_bal['subreddit'].str.contains('|'.join(news_politics_keywords), case=False)]

# # Plot countplot for all filtered subreddits combined
# plt.figure(figsize=(12, 8))
# sns.countplot(x='subreddit', hue='label', data=news_politics_comments, palette='Set1')
# plt.title('Distribution of Sarcastic and Non-sarcastic Comments for News/Politics Subreddits')
# plt.xlabel('Subreddit')
# plt.ylabel('Count')
# plt.xticks(rotation=90)
# plt.legend(title='Label', labels=['Non-Sarcastic', 'Sarcastic'])
# plt.show()


In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Assuming 'train_bal' is your DataFrame and 'label' indicates sarcastic/non-sarcastic comments

# # Define keywords related to news or politics
# news_politics_keywords = ['news', 'politics', 'worldnews', 'worldpolitics', 'uspolitics', 'worldnews', 'worldpolitics']

# # Filter DataFrame for subreddits containing keywords
# news_politics_comments = train_bal[train_bal['subreddit'].str.contains('|'.join(news_politics_keywords), case=False)]

# # Plot the count of sarcastic and non-sarcastic comments for subreddits related to news or politics
# plt.figure(figsize=(10, 6))
# sns.countplot(x='label', data=news_politics_comments, palette='Set1')
# plt.title('Distribution of Sarcastic and Non-sarcastic Comments for News/Politics Subreddits')
# plt.xlabel('Label')
# plt.ylabel('Count')
# plt.xticks(ticks=[0, 1], labels=['Non-Sarcastic', 'Sarcastic'])
# plt.show()

In [ ]:
import pandas as pd

# Assuming 'train_bal' is your DataFrame and 'label' indicates sarcastic/non-sarcastic comments

# Calculate the distribution of sarcastic and non-sarcastic comments for each subreddit
subreddit_counts = train_bal.groupby('subreddit')['label'].value_counts(normalize=True).unstack().fillna(0)

# Define bins for categorizing subreddits based on the percentage of sarcastic comments
bins = [0.1 * i for i in range(11)]  # 0.0, 0.1, ..., 1.0

# Map each category to the midpoint of its range as a numeric value between 0 and 1
subreddit_numeric_category = pd.cut(subreddit_counts[1], bins=bins, labels=[0.05 + 0.1 * i for i in range(10)], include_lowest=True)

# Label remaining subreddits as 'other' and update the DataFrame
train_bal['subreddit_sarcasm_numeric'] = train_bal['subreddit'].map(subreddit_numeric_category).fillna(0.5)  # Fill NaN with 0.5 for 'other' subreddits

# Define keywords related to political subreddits
political_keywords = ['politics', 'political', 'government', 'elections', 'congress', 'policy', 'politicalhumor', 'news', 'uspolitics', 'worldnews', 'worldpolitics']

# Create a new column indicating whether a subreddit is political or not
train_bal['political_subreddit'] = train_bal['subreddit'].str.contains('|'.join(political_keywords), case=False)

# Convert the boolean values to categorical (0 for non-political, 1 for political)
train_bal['political_subreddit'] = train_bal['political_subreddit'].astype(int)

train_bal.info()

In [ ]:
train_bal.political_subreddit.value_counts()

In [ ]:
train_bal.head()

In [ ]:
train_bal['subreddit_sarcasm_numeric'] = train_bal.subreddit_sarcasm_numeric.astype('float64')
train_bal.info()

In [ ]:
# train_bal.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

train_bal['date'] = pd.to_datetime(train_bal['date']) # Run this to convert date to date_time
train_bal['created_utc'] = pd.to_datetime(train_bal['created_utc']) # Run this to convert created_utc to date_time
train_bal['hour'] = train_bal['created_utc'].dt.hour # Run this to feature engineer hour variable
train_bal['month'] = train_bal['date'].dt.month # Run this cell to add month and day of week variables
train_bal['dow'] = train_bal['date'].dt.day_of_week
train_bal.info()

In [ ]:
# Drop unneeded columns
train = train_bal.drop(columns=['Unnamed: 0', 'date', 'created_utc', 'author', 'ups', 'downs', 'score'])
train.info()

In [ ]:
X = train.drop(columns=['label', 'comment', 'parent_comment'])
y = train['label']

In [ ]:
sc = StandardScaler() # Scale variables

X = sc.fit_transform(X)
X = pd.DataFrame(X)
train_cat = pd.concat([train, X], axis=1)

In [ ]:
train_cat.drop(columns=['score', 'hour', 'month', 'dow', 'label', 'subreddit_sarcasm_numeric', 'political_subreddit'], inplace=True)
# train_cat.rename(columns={'0':'score', '1':'ups', '2':'downs', '3':'hour', '4':'month', '5':'dow'}, inplace=True)
train_cat.info()

In [ ]:
# subreddit_counts = train_bal.groupby('subreddit').size().sort_values(ascending=False)
# subreddit_counts.head()

In [ ]:
# # Calculate the 5% quantile
# quantile_5 = subreddit_counts.quantile(0.98)

# # Select subreddits whose counts are greater than or equal to the 5% quantile
# top_subreddits = subreddit_counts[subreddit_counts >= quantile_5]
# top_subreddits.size # 680 subreddits in the top 5%

In [ ]:
# subreddits = top_subreddits.index.tolist()
# subreddits[:5]

In [ ]:
# def filter_subreddit(subreddit):
#     if subreddit in subreddits:
#         return subreddit
#     else:
#         return 'other'

# # Apply the filter_subreddit function to the 'subreddit' column
# train['subreddit'] = train['subreddit'].apply(filter_subreddit)

# train['subreddit'].nunique()

In [ ]:
# top_subreddits = train['subreddit'].value_counts(ascending=False).head(10).index
# subreddit_filter = train[train['subreddit'].isin(top_subreddits)]

# subreddit_filter.info()

In [ ]:
# # One-hot encode subreddit
# subreddit_dummies = pd.get_dummies(train['subreddit'])

In [ ]:
# train['subreddit'].nunique()

In [ ]:
# # Combine one-hot encoded categorical variables with numeric variables
# train_encoded = pd.concat([train_cat, subreddit_dummies], axis=1)
# train_encoded.head()

In [ ]:
# train_encoded.drop(columns=['subreddit'], inplace=True)

In [ ]:
train_encoded = train_cat
train_encoded.shape

In [ ]:
train_encoded.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack

tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False)

comment_vectors = tf_idf_comment.fit_transform(train_encoded['comment'])
parent_comment_vectors = tf_idf_parent_comment.fit_transform(train_encoded['parent_comment'])

# Concatenate the TF-IDF vectors horizontally
combined_vectors = hstack([comment_vectors, parent_comment_vectors])

## Stacking with parent comment

In [ ]:
# Get other variables from train_encoded
other_variables = train_encoded.drop(columns=['comment', 'parent_comment'])

# Concatenate other variables horizontally with TF-IDF vectors
stacked_vars = hstack([combined_vectors, other_variables])

In [ ]:
# Split the data into train and test sets
X_train, X_val, y_train, y_val = train_test_split(stacked_vars, y, test_size=0.2, random_state=42)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
# Split the data into train and test sets
X_train, X_val, y_train, y_val = train_test_split(stacked_vars, y, test_size=0.2, random_state=42)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform

param_grid = {
    'C': uniform(loc=0, scale=4),
}

logistic_regression = LogisticRegression()

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=logistic_regression, param_distributions=param_grid, n_iter=2,
                                   scoring='accuracy', cv=5, verbose=1, random_state=22, n_jobs=-1)

# Fit the random search
random_search.fit(X_train, y_train)

# Print the best parameters and score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

## Subreddit TFIDF Vectorization
Since we've been unsuccessful in one-hot encoding the subreddit feature because of the dimensionality addition, we will try the tf-idf vectorization.

In [ ]:
import pandas as pd

train_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Train_Balanced.csv')   # Make sure path is correct in your google drive
train_bal.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

train_bal['date'] = pd.to_datetime(train_bal['date']) # Run this to convert date to date_time
train_bal['created_utc'] = pd.to_datetime(train_bal['created_utc']) # Run this to convert created_utc to date_time
train_bal['hour'] = train_bal['created_utc'].dt.hour # Run this to feature engineer hour variable
train_bal['month'] = train_bal['date'].dt.month # Run this cell to add month and day of week variables
train_bal['dow'] = train_bal['date'].dt.day_of_week
train_bal.info()

In [ ]:
# Drop unneeded columns
train = train_bal.drop(columns=['Unnamed: 0', 'date', 'created_utc', 'author', 'ups', 'downs'])
train.info()

In [ ]:
X = train.drop(columns=['label', 'comment', 'parent_comment', 'subreddit']) # Drop non-numeric columns
y = train['label']

In [ ]:
sc = StandardScaler() # Scale variables

X = sc.fit_transform(X)
X = pd.DataFrame(X)
train_cat = pd.concat([train, X], axis=1) # concatenate text and scaled numeric variables

In [ ]:
train_cat.drop(columns=['score', 'hour', 'month', 'dow', 'label'], inplace=True) # Drop unscaled variables
# train_cat.rename(columns={'0':'score', '1':'ups', '2':'downs', '3':'hour', '4':'month', '5':'dow'}, inplace=True)
train_cat.info()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack

tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # tf-idf vectors for comment
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # same for parent comment
tf_idf_subreddits = TfidfVectorizer(ngram_range=(1, 1)) # now include subreddit

comment_vectors = tf_idf_comment.fit_transform(train_bal['comment'])
parent_comment_vectors = tf_idf_parent_comment.fit_transform(train_bal['parent_comment'])
subreddit_vectors = tf_idf_subreddits.fit_transform(train_bal['subreddit'])

# Concatenate the TF-IDF vectors horizontally
combined_vectors = hstack([comment_vectors, parent_comment_vectors, subreddit_vectors])

In [ ]:
print(f'Comment TF-IDF Vector shape: {comment_vectors.shape}')
print(f'Parent Comment TF-IDF Vector shape: {parent_comment_vectors.shape}')
print(f'Subreddit TF-IDF Vector shape: {subreddit_vectors.shape}')
print(f'Combined TF-IDF Vector shape: {combined_vectors.shape}')

In [ ]:
# Split the data into train and test sets
X_train, X_val, y_train, y_val = train_test_split(combined_vectors, y, test_size=0.2, random_state=42)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

### Hyperparameter Tuning (C)

Tune C value, cross-validation hyperparameter tuning.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(combined_vectors, y, test_size=0.2, random_state=42)

param_grid = {
    'C': uniform(loc=0, scale=4),
}

logistic_regression = LogisticRegression(solver='lbfgs')

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=logistic_regression, param_distributions=param_grid, n_iter=2,
                                   scoring='accuracy', cv=5, verbose=1, random_state=22, n_jobs=-1)

# Fit the random search
random_search.fit(X_train, y_train)

# Print the best parameters and score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

In [ ]:
# Now we will predict the validation data using the best C value found from our randomized search
best_C = 0.8338421494353705

logit = LogisticRegression(C=best_C, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

We squeezed a bit more accuracy out of our model, from 72.54% to 72.56%

# Test Performance

In [ ]:
test_bal = pd.read_csv('/content/drive/MyDrive/Sarcasm_Data/Test_Balanced.csv')
test_bal.info()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

test_bal['date'] = pd.to_datetime(test_bal['date']) # Run this to convert date to date_time
test_bal['created_utc'] = pd.to_datetime(test_bal['created_utc']) # Run this to convert created_utc to date_time
test_bal['hour'] = test_bal['created_utc'].dt.hour # Run this to feature engineer hour variable
test_bal['month'] = test_bal['date'].dt.month # Run this cell to add month and day of week variables
test_bal['dow'] = test_bal['date'].dt.day_of_week
test_bal.info()

In [ ]:
# Drop unneeded columns
test = test_bal.drop(columns=['Unnamed: 0', 'date', 'created_utc', 'author', 'ups', 'downs'])
test.info()

In [ ]:
X = test.drop(columns=['label', 'comment', 'parent_comment', 'subreddit']) # Drop non-numeric columns
y = test['label']

In [ ]:
sc = StandardScaler() # Scale variables

X = sc.fit_transform(X)
X = pd.DataFrame(X)
test_cat = pd.concat([test, X], axis=1) # concatenate text and scaled numeric variables

In [ ]:
test_cat.drop(columns=['score', 'hour', 'month', 'dow', 'label'], inplace=True) # Drop unscaled variables
# train_cat.rename(columns={'0':'score', '1':'ups', '2':'downs', '3':'hour', '4':'month', '5':'dow'}, inplace=True)
test_cat.info()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack

tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # tf-idf vectors for comment
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # same for parent comment
tf_idf_subreddits = TfidfVectorizer(ngram_range=(1, 1)) # now include subreddit

comment_vectors = tf_idf_comment.fit_transform(test_bal['comment'])
parent_comment_vectors = tf_idf_parent_comment.fit_transform(test_bal['parent_comment'])
subreddit_vectors = tf_idf_subreddits.fit_transform(test_bal['subreddit'])

# Concatenate the TF-IDF vectors horizontally
combined_vectors = hstack([comment_vectors, parent_comment_vectors, subreddit_vectors])

In [ ]:
print(f'Comment TF-IDF Vector shape: {comment_vectors.shape}')
print(f'Parent Comment TF-IDF Vector shape: {parent_comment_vectors.shape}')
print(f'Subreddit TF-IDF Vector shape: {subreddit_vectors.shape}')
print(f'Combined TF-IDF Vector shape: {combined_vectors.shape}')

In [ ]:
# Split the data into train and test sets
X_train, X_val, y_train, y_val = train_test_split(combined_vectors, y, test_size=0.2, random_state=42)

logit = LogisticRegression(C=1, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

### Hyperparameter Tuning (C)

Tune C value, cross-validation hyperparameter tuning.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(combined_vectors, y, test_size=0.2, random_state=42)

param_grid = {
    'C': uniform(loc=0, scale=4),
}

logistic_regression = LogisticRegression(solver='lbfgs')

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=logistic_regression, param_distributions=param_grid, n_iter=2,
                                   scoring='accuracy', cv=5, verbose=1, random_state=22, n_jobs=-1)

# Fit the random search
random_search.fit(X_train, y_train)

# Print the best parameters and score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

In [ ]:
# Now we will predict the validation data using the best C value found from our randomized search
best_C = 0.8338421494353705

logit = LogisticRegression(C=best_C, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(X_train, y_train)
y_pred = logit.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_val, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_val, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_val, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_val, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_val, y_pred)
print('ROC AUC: %f' % auc)

# Test

In [ ]:
X_train = train_bal[['comment', 'parent_comment', 'subreddit']]
y_train = train_bal['label']
X_test = test[['comment', 'parent_comment', 'subreddit']]
y_test = test['label']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack

tf_idf_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # tf-idf vectors for comment
tf_idf_parent_comment = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=False) # same for parent comment
tf_idf_subreddits = TfidfVectorizer(ngram_range=(1, 1)) # now include subreddit

# Fit the TfidfVectorizer on the training data and transform the training data
comment_vectors_train = tf_idf_comment.fit_transform(X_train['comment'])
parent_comment_vectors_train = tf_idf_parent_comment.fit_transform(X_train['parent_comment'])
subreddit_vectors_train = tf_idf_subreddits.fit_transform(X_train['subreddit'])

# Concatenate the TF-IDF vectors horizontally for training data
combined_vectors_train = hstack([comment_vectors_train, parent_comment_vectors_train, subreddit_vectors_train])

# Transform the testing data based on the vocabulary learned from the training data
comment_vectors_test = tf_idf_comment.transform(X_test['comment'])  # Note the change here to transform
parent_comment_vectors_test = tf_idf_parent_comment.transform(X_test['parent_comment'])  # And here
subreddit_vectors_test = tf_idf_subreddits.transform(X_test['subreddit'])  # And here

# Concatenate the TF-IDF vectors horizontally for testing data
combined_vectors_test = hstack([comment_vectors_test, parent_comment_vectors_test, subreddit_vectors_test])

In [ ]:
# Now we will predict the validation data using the best C value found from our randomized search
best_C = 0.8338421494353705

logit = LogisticRegression(C=best_C, n_jobs=-1, solver='lbfgs',
                           random_state=22, verbose=1)

logit.fit(combined_vectors_train, y_train)
y_pred = logit.predict(combined_vectors_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
print('Accuracy: %f' % accuracy)

precision = precision_score(y_test, y_pred)
print('Precision: %f' % precision)

recall = recall_score(y_test, y_pred)
print('Recall: %f' % recall)

f1 = f1_score(y_test, y_pred)
print('F1 score: %f' % f1)

auc = roc_auc_score(y_test, y_pred)
print('ROC AUC: %f' % auc)